## 4.2 Synthesis Dashboard

Ce tableau de bord interactif rassemble les 4 indicateurs clés du marché IT marocain :
1. **Répartition géographique** (Volume d'offres par ville)
2. **Top 15 des compétences** les plus demandées
3. **Distribution des salaires** pour les profils Data (Scientist, Engineer, Analyst)
4. **Évolution mensuelle** des offres pour la famille Data depuis Janvier 2023

In [4]:
# Code interactif du Dashboard de synthèse Mexora RH
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Connexion à DuckDB
con = duckdb.connect()
gold_dir = 'data_lake_mexora_rh/gold'
silver_path = 'data_lake_mexora_rh/silver/offres_clean/offres_clean.parquet'

# ---------------------------------------------------------
# 1. CARTE DU MAROC : Volume d'offres par ville
# ---------------------------------------------------------
df_ville_volume = con.execute(f"""
    SELECT ville, SUM(nb_offres) AS volume
    FROM read_parquet('{gold_dir}/offres_par_ville.parquet')
    WHERE ville IN ('Casablanca', 'Rabat', 'Tanger', 'Marrakech', 'Fès')
    GROUP BY ville
""").df()

# Coordonnées des villes marocaines
coords = {
    'Casablanca': {'lat': 33.5731, 'lon': -7.5898},
    'Rabat': {'lat': 34.0209, 'lon': -6.8416},
    'Tanger': {'lat': 35.7595, 'lon': -5.8340},
    'Marrakech': {'lat': 31.6295, 'lon': -7.9811},
    'Fès': {'lat': 34.0331, 'lon': -5.0003}
}
df_ville_volume['lat'] = df_ville_volume['ville'].map(lambda x: coords[x]['lat'])
df_ville_volume['lon'] = df_ville_volume['ville'].map(lambda x: coords[x]['lon'])

fig_map = px.scatter_geo(
    df_ville_volume,
    lat='lat',
    lon='lon',
    size='volume',
    hover_name='ville',
    text='ville',
    size_max=40,
    projection='mercator',
    title='Volume d\'offres IT par ville au Maroc'
)
fig_map.update_geos(
    center=dict(lat=32.0, lon=-6.0),
    projection_scale=6,
    showland=True,
    landcolor='lightgray',
    showocean=True,
    oceancolor='azure'
)
fig_map.update_traces(textposition='top center')

# ---------------------------------------------------------
# 2. TOP 15 COMPÉTENCES : Horizontal bar chart par famille
# ---------------------------------------------------------
df_skills = con.execute(f"""
    SELECT 
        famille, 
        competence,
        CAST(SUM(nb_offres_mentionnent) AS INTEGER) AS mentions,
        ROUND(SUM(pct_offres_total), 2) AS pourcentage
    FROM read_parquet('{gold_dir}/top_competences.parquet')
    GROUP BY famille, competence
    ORDER BY mentions DESC
    LIMIT 15
""").df()

fig_skills = px.bar(
    df_skills.sort_values('mentions', ascending=True),
    x='pourcentage',
    y='competence',
    color='famille',
    orientation='h',
    title='Top 15 des compétences IT les plus demandées',
    labels={'pourcentage': 'Pourcentage des offres (%)', 'competence': 'Compétence'}
)

# ---------------------------------------------------------
# 3. BOXPLOT SALAIRES : Distribution par profil
# ---------------------------------------------------------
df_salaries_raw = con.execute(f"""
    SELECT profil_normalise AS profil, salaire_median_mad
    FROM '{silver_path}'
    WHERE salaire_connu = true
      AND profil_normalise IN ('Data Scientist', 'Data Engineer', 'Data Analyst')
""").df()

fig_salaries = px.box(
    df_salaries_raw,
    x='profil',
    y='salaire_median_mad',
    color='profil',
    points='outliers',
    title='Distribution des salaires par profil Data (MAD/mois)',
    labels={'salaire_median_mad': 'Salaire Brut (MAD)', 'profil': 'Profil'}
)
fig_salaries.update_layout(showlegend=False)

# ---------------------------------------------------------
# 4. ÉVOLUTION MENSUELLE : Tendances par profil Data
# ---------------------------------------------------------
df_trends = con.execute(f"""
    SELECT annee || '-' || mois AS date_val, profil, SUM(nb_offres) AS count
    FROM read_parquet('{gold_dir}/offres_par_ville.parquet')
    WHERE profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
    GROUP BY annee, mois, profil
    ORDER BY date_val, profil
""").df()

fig_trends = px.line(
    df_trends,
    x='date_val',
    y='count',
    color='profil',
    title='Évolution mensuelle des offres par profil Data',
    labels={'date_val': 'Mois', 'count': 'Nombre d\'offres', 'profil': 'Profil'},
    markers=True
)

# Afficher les visualisations
fig_map.show()
fig_skills.show()
fig_salaries.show()
fig_trends.show()
con.close()